# Notebook 21 — ISLES2024 WU-20B: Full-fold normalization & anatomical-mask QC

**CPU-only, read-only-source, resumable.** This notebook audits image-only NCCT tissue support across **149 patients**, creates private overlays for anatomical review, and fits **five fold-specific candidate parameter manifests** using only each fold's 119/120 training images. All images, samples, and case IDs remain in your **private** Google Drive. Do not upload patient-derived images/parameters to public GitHub.

**Training remains blocked even on successful completion.** The existing mask is a **provisional head-tissue proxy, not a clinically validated brain mask**; the q0.5/q99.5 clipping and ±8 robust-z ceiling are only unapproved research candidates. Nothing here stages Datasets506/507 for training, installs the model, or launches any GPU jobs. Dataset504/505 remain frozen.

**Runtime:** CPU; use Runtime → Run all. The script saves a small per-case checkpoint to Drive; if the Colab session ends, rerun the notebook to resume. Outputs: `MyDrive/HybridStrokeSeg/ISLES2024/results/wu20b_full_fold_calibration/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Verify immutable data and fetch audited code

Input split is frozen by SHA256. Fetch the script and the exact WU19/WU20 modules from a pinned GitHub commit rather than mutable `main`.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
from datetime import datetime, timezone

ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
RAW = ROOT/'nnUNet_raw'/'Dataset505_ISLES2024_NCCT_PERFUSION'
SPLITS = ROOT/'nnUNet_preprocessed'/'Dataset505_ISLES2024_NCCT_PERFUSION'/'splits_final.json'
PLANS = ROOT/'nnUNet_preprocessed'/'Dataset505_ISLES2024_NCCT_PERFUSION'/'nnUNetPlans.json'
OUTPUT = ROOT/'results'/'wu20b_full_fold_calibration'
LOG = OUTPUT/'WU20B_full_fold_calibration.log'
CODE_COMMIT = 'e19eeb49096153b0985cea9733c3f764bb84c5f8'
SOURCE = Path('/content/isles2024_wu20b_source')
PKG = SOURCE/'src'/'hybridstrokeseg'
PKG.mkdir(parents=True, exist_ok=True)
(SOURCE/'scripts').mkdir(parents=True, exist_ok=True)

EXPECTED_SPLIT_SHA = '664ed51e15374f8f9568d586fba2cb87d5157dc90986e5eca82c9c8440bf6027'
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as fp:
        for b in iter(lambda: fp.read(1024*1024), b''):h.update(b)
    return h.hexdigest()
assert RAW.is_dir(), RAW
assert SPLITS.is_file() and PLANS.is_file()
assert sha256(SPLITS) == EXPECTED_SPLIT_SHA
BASE = f'https://raw.githubusercontent.com/NTinkicht/HybridStrokeSeg-Reproduction/{CODE_COMMIT}'
files = {
 'src/hybridstrokeseg/wu19_normalization_protocol.py':PKG/'wu19_normalization_protocol.py',
 'src/hybridstrokeseg/wu20_fold_preprocessing.py':PKG/'wu20_fold_preprocessing.py',
 'scripts/calibrate_isles2024_all_folds.py':SOURCE/'scripts'/'calibrate_isles2024_all_folds.py',
}
for relative, dest in files.items():
    subprocess.run(['curl','-fLsS','--retry','5','--retry-delay','3',f'{BASE}/{relative}','-o',str(dest)], check=True)
    assert dest.stat().st_size > 1000, f'Incomplete source: {relative}'
    print('Fetched pinned code:',relative, sha256(dest)[:15])
print('Source commit:',CODE_COMMIT)
print('Frozen splits:',sha256(SPLITS))


## 2. Install CPU dependencies and confirm candidate support mask

No nnU-Net GPU training dependencies are needed. The mask below is only a deterministic *proxy* for an independent review, not an approved brain mask.


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','--only-binary=:all:',
 'numpy>=1.26,<3','scipy>=1.12,<2','nibabel>=5.2,<6','matplotlib>=3.8,<4'],check=True)
sys.path.insert(0,str(SOURCE/'src'))
from hybridstrokeseg.wu20_fold_preprocessing import ncct_head_tissue_support
import numpy as np
sample = np.full((16,32,32),-1000.,dtype=np.float32)
sample[3:14,5:27,5:27] = 40.
mask,mask_qc = ncct_head_tissue_support(sample)
assert mask.sum() == 11*22*22
assert mask_qc['clinical_brain_mask_validated'] is False
print('Synthetic mask function smoke: PASS')


## 3. Resumable full-cohort image-only calibration (149 patients)

Full run reads all original acute channels once per patient (and reloads selected NCCT images for private overlays). It records one atomic checkpoint per case, then fits each of the five training-only folds, without accessing validation labels/outcomes. If interrupted, rerunning skips verified unchanged checkpoints. A partial result is **not** training-ready. The full run may need more than one Colab session; do not remove checkpoints.


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
COMMAND = [sys.executable,'-u',str(SOURCE/'scripts'/'calibrate_isles2024_all_folds.py'),
 '--raw505',str(RAW),'--splits',str(SPLITS),'--plans505',str(PLANS),
 '--output-dir',str(OUTPUT)]
env = os.environ.copy()
env['PYTHONPATH'] = str(SOURCE/'src') + os.pathsep + env.get('PYTHONPATH','')
print('Running image-only full-fold calibration. Outputs are private:',OUTPUT,flush=True)
with LOG.open('a',encoding='utf8') as log:
    log.write('\n'+datetime.now(timezone.utc).isoformat()+' WU20B pinned '+CODE_COMMIT+'\n')
    with subprocess.Popen(COMMAND,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          text=True,bufsize=1) as proc:
        for line in proc.stdout:
            print(line,end='',flush=True)
            log.write(line); log.flush()
        rc=proc.wait()
if rc:
    status_path = OUTPUT/'WU20B_progress.json'
    print('Partial run or QC failure. Rerun the notebook to resume. See:',status_path)
    raise RuntimeError(f'WU20B stopped (exit={rc}); inspect {status_path} and {LOG}')


## 4. Confirm train-only folds and review private anatomical overlays

A complete calibration is **still not a model release**. Open the mask overlay PNGs in Google Drive for independent anatomical QC, especially `sub_stroke0016`; all overlays are based only on NCCT images, never lesion labels. Do not tune boundaries based on validation Dice.


In [ ]:
progress=json.loads((OUTPUT/'WU20B_progress.json').read_text(encoding='utf8'))
report=json.loads((OUTPUT/'WU20B_aggregate.json').read_text(encoding='utf8'))
assert progress['status']=='COMPLETE' and progress['completed_verified']==149, progress
assert report['status']=='FULL_FOLD_PROVISIONAL_FIT_COMPLETE', report
assert report['n_patients']==149 and report['n_folds']==5, report
assert report['training_ready'] is False
for fold in report['folds']:
    assert fold['n_train'] in (119,120) and fold['n_heldout'] in (29,30)
print('All 149/149 source cases and 5 train-only fits complete.')
print('QC aggregate:',report['qc_aggregate'])
print('Candidate fold manifests:',OUTPUT/'fold_private')
print('Private mask overlay images:',OUTPUT/'private_mask_overlays')
print('Architecture lock proposal (NOT qualified):',OUTPUT/'WU20B_proposed_architecture_lock_PRIVATE.json')
print('GPU training: STILL BLOCKED. Manual mask QC + nnU-Net inference qualification required.')
